# LinguaFranca - Phase 1: Dataset Construction

### Confirmed working configuration
- Model: `meta-llama/Llama-3.2-3B-Instruct`
- Benchmark: Tag Well-Formed Rate **100%** | Natural Failure Rate **28.8%**
- NNsight 0.7.x activation patching: confirmed (two-separate-traces pattern)
- Auto-checkpoints to HuggingFace every 100 examples

**Resuming after timeout:** Re-run cells 2 to 6. Already-processed IDs are skipped automatically.


## Step 0 - Check GPU

In [ ]:
import subprocess, torch
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True)
print('GPU:', r.stdout.strip() or 'NOT FOUND - enable GPU in Notebook Settings!')
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')


## Step 1 - Install Dependencies

In [ ]:
%%capture
!pip install nnsight==0.7.0 sentence-transformers SPARQLWrapper datasets accelerate huggingface_hub -q


## Step 2 - Clone / Pull Source Code

Always pulls the latest version. Re-run after a timeout to get latest fixes.


In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/ANLPproject/LinguaFranca.git'
WORK_DIR = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK_DIR):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', REPO_URL, WORK_DIR], check=True)
else:
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=True)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())


## Step 3 - Set HuggingFace Credentials

- `HF_TOKEN`: needed to download gated Llama 3.2 AND to upload checkpoints
- `HF_REPO_ID`: your HF dataset repo, e.g. `Anish/LinguaFranca-Phase1`


In [ ]:
import os
HF_TOKEN   = 'hf_your_token_here'              # paste your write token here
HF_REPO_ID = 'your_username/LinguaFranca-Phase1'  # paste your repo here
os.environ['HF_TOKEN']   = HF_TOKEN
os.environ['HF_REPO_ID'] = HF_REPO_ID
from huggingface_hub import whoami
try:
    u = whoami(token=HF_TOKEN)
    print('Logged in as:', u['name'], '| checkpoints ->', HF_REPO_ID)
except Exception as e:
    print('HF token error:', e, '| generation still works but no checkpoints')


## Step 4 - Load Config

Config is already set to:
- Model: `meta-llama/Llama-3.2-3B-Instruct`
- n_source_examples: 2000
- Context: gold supporting-fact passages only (not first-N paragraphs)
- Question filter: compositional only (comparison questions excluded)


In [ ]:
import yaml, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s')
with open('configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['raw_dir']           = '/kaggle/working/data/raw'
cfg['data']['processed_dir']     = '/kaggle/working/data/processed'
cfg['data']['hidden_states_dir'] = '/kaggle/working/data/hidden_states'
cfg['matching']['wikidata_aliases_path'] = '/kaggle/working/data/raw/wikidata_aliases.json'
with open('configs/data_config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('Model           :', cfg['model']['name'])
print('n_source_examples:', cfg['data']['n_source_examples'])
print('Layers          :', cfg['hidden_states']['layers'])
print('SBERT threshold :', cfg['matching']['sbert_threshold'])
print('raw_dir         :', cfg['data']['raw_dir'])


## Step 5 - Verify NNsight Write-Access

Runs the confirmed two-separate-traces activation patching pattern (approx 30s).
If it prints the patched logits shape, proceed to Step 6.


In [ ]:
import torch, gc, os
from nnsight import LanguageModel
print('Loading Llama 3.2 3B via NNsight...')
nn_model = LanguageModel(
    'meta-llama/Llama-3.2-3B-Instruct',
    device_map='auto',
    torch_dtype=torch.float16,
    token=os.environ.get('HF_TOKEN', ''),
)
# Trace 1: capture hidden state from source prompt
with nn_model.trace('The capital of France is Paris'):
    saved_hs = nn_model.model.layers[0].output[0].save()
source_tensor = saved_hs.value if hasattr(saved_hs, 'value') else saved_hs
print('Source tensor shape:', source_tensor.shape)
# Trace 2: inject real tensor into target prompt
with nn_model.trace('The capital of Germany is Berlin'):
    nn_model.model.layers[0].output[0] = source_tensor
    patched_logits = nn_model.lm_head.output.save()
result = patched_logits.value if hasattr(patched_logits, 'value') else patched_logits
print('Patched logits shape:', result.shape)
print('NNsight write-access confirmed!')
del nn_model; gc.collect(); torch.cuda.empty_cache()
print('VRAM cleared. Ready for Step 6.')


## Step 6 - Generate CoT + Extract Hidden States

- Filters out comparison questions, builds gold-title context
- Checkpoints to HF every 100 examples
- Resumes automatically if you restart (already-processed IDs are skipped)

> Estimated time: approx 2 hours for 2000 examples on T4 x1


In [ ]:
import os, json
from phase1_dataset.generate_cot import run_generation
cot_path = run_generation(cfg, hf_token=os.environ.get('HF_TOKEN'), dry_run=False)
with open(cot_path) as f:
    records = [json.loads(l) for l in f]
wf = sum(1 for r in records if r.get('hop_spans'))
print('CoT generation complete!')
print('Total generated    :', len(records))
print('Tag well-formed    :', wf)
print('Discarded (no tags):', len(records) - wf)


## Step 7 - Label Hops (Success / Failure)

Uses bipartite order-independent matching (SBERT threshold=0.70, LLM judge fallback).
Empty gold entities are correctly skipped (label=-1, not counted as failures).


In [ ]:
import json, os
import torch
from pathlib import Path
from utils.matching import EntityMatcher
from utils.wikidata_aliases import load_alias_table
from phase1_dataset.label_hops import label_example, make_llm_judge
from phase1_dataset.generate_cot import load_model_and_tokenizer
cot_path = Path(cfg['data']['raw_dir']) / '2wikimultihopqa' / 'generated_cot.jsonl'
with open(cot_path) as f:
    records = [json.loads(l) for l in f if l.strip()]
print('Loaded', len(records), 'CoT records')
model, tokenizer = load_model_and_tokenizer(cfg)
match_cfg = cfg['matching']
aliases = load_alias_table(match_cfg['wikidata_aliases_path'])
matcher = EntityMatcher(
    aliases=aliases,
    sbert_model_name=match_cfg['sbert_model'],
    sbert_threshold=match_cfg['sbert_threshold'],
    llm_judge=make_llm_judge(model, tokenizer),
    llm_budget=match_cfg['llm_fallback_budget'],
)
labeled = [label_example(ex, matcher) for ex in records]
total_hops = sum(1 for ex in labeled for h in ex.get('hops', []) if h['label'] != -1)
fail_hops  = sum(1 for ex in labeled for h in ex.get('hops', []) if h['label'] == 1)
print('Total valid hops :', total_hops)
print('Failed hops      :', fail_hops)
print('Success hops     :', total_hops - fail_hops)
labeled_path = Path(cfg['data']['raw_dir']) / '2wikimultihopqa' / 'labeled.jsonl'
with open(labeled_path, 'w') as f:
    for ex in labeled: f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print('Labeled data saved to:', labeled_path)
matcher.llm_judge = None
del model, tokenizer
import gc; gc.collect(); torch.cuda.empty_cache()


## Step 8 - Layer Probing (Find Best Layers)

Trains a logistic probe per layer. Note the suggested BEST_LAYERS and
re-run Step 6 with only those layers to save storage and time.


In [ ]:
import json, torch, numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
hs_dir = Path(cfg['data']['hidden_states_dir'])
with open(Path(cfg['data']['raw_dir']) / '2wikimultihopqa' / 'labeled.jsonl') as f:
    examples = [json.loads(l) for l in f]
pt_files = list(hs_dir.glob('*.pt'))
if not pt_files:
    print('No .pt files found - check hidden_states.extract=true in config')
else:
    sample = torch.load(pt_files[0])
    layer_indices = sample.get('layer_indices', list(range(sample['pooled'].shape[0])))
    X_by_layer = {li: [] for li in layer_indices}
    y = []
    for ex in examples:
        pt_path = hs_dir / (ex['id'] + '.pt')
        if not pt_path.exists(): continue
        data = torch.load(pt_path)
        pooled = data['pooled']
        li_list = data.get('layer_indices', list(range(pooled.shape[0])))
        for li_idx, li in enumerate(li_list):
            X_by_layer[li].append(pooled[li_idx].mean(0).detach().numpy())
        y.append(1 if ex.get('first_fail_hop') is not None else 0)
    y = np.array(y)
    print('Features:', len(y), 'examples | failure ratio:', round(100*float(y.mean()), 1), 'pct')
    layer_accs = {}
    for li in layer_indices:
        X = np.nan_to_num(np.array(X_by_layer.get(li, [])))
        if len(X) == 0 or len(np.unique(y)) < 2: continue
        clf = LogisticRegression(max_iter=500)
        clf.fit(X, y)
        layer_accs[li] = accuracy_score(y, clf.predict(X))
    print('\nLayer probe accuracy:')
    for li, acc in sorted(layer_accs.items()):
        print('  Layer', str(li).rjust(2), ':', round(acc, 3), chr(0x2588)*int(acc*40))
    best = sorted(layer_accs, key=layer_accs.get, reverse=True)[:8]
    print('\n-> Suggested BEST_LAYERS =', sorted(best))
    print('Set cfg[hidden_states][layers] to these and re-run Step 6.')


## Step 9 - Build Counterfactuals (Class Balancing)

In [ ]:
import os, json
from phase1_dataset.counterfactuals import run_counterfactuals
augmented_path = run_counterfactuals(cfg, hf_token=os.environ.get('HF_TOKEN'))
with open(augmented_path) as f:
    all_ex = [json.loads(l) for l in f]
n_orig = sum(1 for e in all_ex if not e.get('is_counterfactual', False))
n_cf   = sum(1 for e in all_ex if e.get('is_counterfactual', False))
print('Original:', n_orig, '| Counterfactual:', n_cf, '| Total:', len(all_ex))


## Step 10 - Write Train / Val / Test Splits

In [ ]:
import json, random
from pathlib import Path
augmented_path = Path(cfg['data']['raw_dir']) / '2wikimultihopqa' / 'augmented.jsonl'
with open(augmented_path) as f:
    all_ex = [json.loads(l) for l in f]
rng = random.Random(cfg.get('seed', 42))
ids = list({e['id'] for e in all_ex}); rng.shuffle(ids)
n = len(ids); t = int(n*0.6); v = int(n*0.2)
tr_ids = set(ids[:t]); va_ids = set(ids[t:t+v]); te_ids = set(ids[t+v:])
train = [e for e in all_ex if e['id'] in tr_ids]
val   = [e for e in all_ex if e['id'] in va_ids]
test  = [e for e in all_ex if e['id'] in te_ids]
assert not (tr_ids & va_ids) and not (tr_ids & te_ids) and not (va_ids & te_ids)
processed = Path(cfg['data']['processed_dir']); processed.mkdir(parents=True, exist_ok=True)
for name, split in [('train', train), ('val', val), ('test', test)]:
    with open(processed / (name + '.jsonl'), 'w') as f:
        for e in split: f.write(json.dumps(e, ensure_ascii=False) + '\n')
    n_fail  = sum(1 for e in split for h in e.get('hops', []) if h['label'] == 1)
    n_total = sum(1 for e in split for h in e.get('hops', []) if h['label'] != -1)
    print(name, ':', len(split), 'examples | hop failure rate:', round(100*n_fail/max(n_total,1), 1), 'pct')
print('No leakage detected.')
print('train:', len(train), '| val:', len(val), '| test:', len(test))


## Step 11 - Inspect a Sample

In [ ]:
import json, random
from pathlib import Path
with open(Path(cfg['data']['processed_dir']) / 'train.jsonl') as f:
    data = [json.loads(l) for l in f]
for ex in random.sample(data, min(3, len(data))):
    print('-' * 70)
    print('Q:', ex['question'])
    print('A:', ex['gold_answer'], '| CF:', ex.get('is_counterfactual', False))
    for h in ex.get('hops', []):
        if h['label'] == -1: continue
        s = 'PASS' if h['label'] == 0 else 'FAIL'
        print('  hop' + str(h['hop_idx']), '[' + s + ']',
              'gold=' + repr(h['bridging_entity_gold']),
              'pred=' + repr(h['bridging_entity_pred']))
        print('    text:', h['text'][:100])
    print('  first_fail_hop:', ex.get('first_fail_hop'))


## Phase 1 Complete!

Output files are in `/kaggle/working/data/processed/`.
Download via **Data - Output** in the Kaggle sidebar.

| File | Contents |
|------|----------|
| `processed/train.jsonl` | 60% training examples with hop labels |
| `processed/val.jsonl` | 20% validation |
| `processed/test.jsonl` | 20% test |
| `hidden_states/<id>.pt` | pooled [n_layers, n_hops, hidden_dim] and single_token dict |

**Next: Phase 2** - train linear probes on hidden states to predict hop failure.
